In [2]:
import torch
dim = 20
rope_base = 1e6
freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
print(freqs)

d:\Python313\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


tensor([1.0000e+00, 2.5119e-01, 6.3096e-02, 1.5849e-02, 3.9811e-03, 1.0000e-03,
        2.5119e-04, 6.3096e-05, 1.5849e-05, 3.9811e-06])


In [10]:

from typing import Optional, Tuple, List, Union

def precompute_freqs(
    dim: int,
    end: int = int(32 * 1024),
    rope_base: float = 1e6,
    rope_scaling: Optional[dict] = None,
):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))

    if rope_scaling is not None:
        original_max, factor, beta_fast, beta_slow = (
            rope_scaling.get("original_max_position_embeddings", 2048),
            rope_scaling.get("factor", 4),
            rope_scaling.get("beta_fast", 4.0),
            rope_scaling.get("beta_slow", 1.0),
        )

        if end / original_max > 1.0:
            corr_dim = next(
                (i for i in range(dim // 2) if 2 * math.pi / freqs[i] > original_max),
                dim // 2,
            )

            power = torch.arange(0, dim // 2, device=freqs.device).float() / max(
                dim // 2 - 1, 1
            )

            beta = beta_slow + (beta_fast - beta_slow) * power

            scale = torch.where(
                torch.arange(dim // 2, device=freqs.device) < corr_dim,
                (beta * factor - beta + 1) / (beta * factor),
                1.0 / factor,
            )

            freqs = freqs * scale

    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()

    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)

    return freqs_cos, freqs_sin


In [13]:
t1 = torch.tensor([[[1,2,3],[4,5,6]],[[13,14,15],[16,17,18]]])
t2 = torch.tensor([[[7,8,9],[10,11,12]],[[19,20,21],[22,23,24]]])
print(t1)
print(t1.shape)

result2 = torch.cat((t1,t2), dim = 1)
print(result2)
print(result2.shape)

tensor([[[ 1,  2,  3],
         [ 4,  5,  6]],

        [[13, 14, 15],
         [16, 17, 18]]])
torch.Size([2, 2, 3])
tensor([[[ 1,  2,  3],
         [ 4,  5,  6],
         [ 7,  8,  9],
         [10, 11, 12]],

        [[13, 14, 15],
         [16, 17, 18],
         [19, 20, 21],
         [22, 23, 24]]])
torch.Size([2, 4, 3])


In [14]:
print(t1.shape[-1])

3
